In [1]:
import anndata as ad

In [2]:
#data_dict = {}
#for i in range(14):
#    id = str(i + 1)
#    data_dict.update({f"plate_{i+1}":f"/home/scratch.sdicks_gpu/data/tahoe/plate{id}_filt_Vevo_Tahoe100M_WServicesFrom_ParseGigalab.h5ad"})
##ad.experimental.concat_on_disk(
#    data_dict,
#    f'/home/scratch.sdicks_gpu/data/tahoe/merge/plate_merged.zarr',
#    label='plate',
#)

In [3]:
import dask
import time
import gc

from dask_cuda import LocalCUDACluster
from dask.distributed import Client

In [4]:
%%time
import cupy as cp
import rapids_singlecell as rsc

preprocessing_gpus="0,1,2,3,4,5,6,7"
cluster = LocalCUDACluster(CUDA_VISIBLE_DEVICES=preprocessing_gpus,
                           threads_per_worker=10,
                           protocol="ucx",
                           rmm_pool_size= "50GB",
                           rmm_maximum_pool_size = "100GB",
                           rmm_allocator_external_lib_list= "cupy",
                          )

client = Client(cluster)

client

2025-11-19 09:18:38 | [INFO] init


CPU times: user 16.2 s, sys: 7 s, total: 23.2 s
Wall time: 35.9 s


Connection method: Cluster object,Cluster type: dask_cuda.LocalCUDACluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 8
Total threads: 80,Total memory: 1.97 TiB
Status: running,Using processes: True
Comm: ucx://127.0.0.1:33146,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: ucx://127.0.0.1:49214,Total threads: 10
Dashboard: http://127.0.0.1:34499/status,Memory: 251.81 GiB
Nanny: ucx://127.0.0.1:51794,


## **Loading Large Datasets into AnnData with Dask**  

To efficiently handle large-scale single-cell datasets, we load data directly from an **HDF5 (`h5`) or Zarr file**  
into an **AnnData object** using **Dask arrays**. This enables **lazy loading**, allowing data to be processed in chunks  
without exceeding memory limits.  

We achieve this using **`read_elem_as_dask`**, which loads the expression matrix (`X`) as a **Dask array**

In [5]:
import rapids_singlecell as rsc
import anndata as ad
import time

In [6]:
from packaging.version import parse as parse_version

if parse_version(ad.__version__) < parse_version("0.12.0rc1"):
    from anndata.experimental import read_elem_as_dask as read_dask
else:
    from anndata.experimental import read_elem_lazy as read_dask
import zarr

SPARSE_CHUNK_SIZE = 20_000
data_pth = "/home/scratch.sdicks_gpu/data/tahoe/merge/plate_merged.zarr" #11Million Cells
#data_pth = "zarr/nvidia_1.3M.zarr" #1.3Million Cells

f = zarr.open(data_pth)
X = f["X"]
shape = X.attrs["shape"]
adata = ad.AnnData(
    X = read_dask(X, (SPARSE_CHUNK_SIZE, shape[1])),
    obs = ad.io.read_elem(f["obs"]),
    var = ad.io.read_elem(f["var"])
)

/tmp/ipykernel_609363/3192190710.py:3: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if parse_version(ad.__version__) < parse_version("0.12.0rc1"):


## **Transferring AnnData to GPU and Persisting Data**  

To leverage **multi-GPU acceleration**, we transfer the AnnData object to GPU memory  
and persist its **Dask-backed expression matrix** for efficient computation. 

**Step-by-Step Breakdown:**
1. Move AnnData to GPU → `rsc.get.anndata_to_GPU(adata)`
    * Transfers all numerical data (`.X`) to GPU memory.
2. Persist the Expression Matrix → `adata.X = adata.X.persist()`
    * Keeps `adata.X` in memory across Dask workers, avoiding redundant recomputation.
3. Optimize Chunking → `adata.X.compute_chunk_sizes()`
    * Computes the exact chunk sizes for optimal Dask scheduling and memory usage.

In [7]:
%%time
#pass_filter_mask = adata.obs["pass_filter"] == "full"
#adata = adata[pass_filter_mask, :].copy()

CPU times: user 11 μs, sys: 3 μs, total: 14 μs
Wall time: 31.2 μs


In [8]:
%%time
rsc.get.anndata_to_GPU(adata)

CPU times: user 153 ms, sys: 53.4 ms, total: 206 ms
Wall time: 167 ms


In [9]:
t1 = time.time()

## **Log Normalization (Fully Lazy Execution)**  

Next, we apply **log normalization** to scale gene expression values.  
This step ensures that differences in sequencing depth across cells do not dominate downstream analysis.  

In [10]:
gc.collect()  

237

In [11]:
%%time
rsc.pp.normalize_total(adata,target_sum = 10000)
rsc.pp.log1p(adata)

CPU times: user 17 ms, sys: 2.79 ms, total: 19.8 ms
Wall time: 17.3 ms


## **Storing the Processed Data in Memory**  

After Log Normalization, we persist the updated expression matrix  
to store the new results in memory for efficient access.  

## **Selecting Highly Variable Genes**  

To focus on the most informative features, we identify **highly variable genes (HVGs)**  
using the **Cell Ranger** method and subset the dataset accordingly.  

* Copy is Essential → Using `.copy()` prevents views, ensuring the operation works properly with Dask-backed AnnData.

In [12]:
%%time
rsc.pp.highly_variable_genes(adata,n_top_genes=5000, flavor="cell_ranger")

CPU times: user 1min 55s, sys: 6.07 s, total: 2min 2s
Wall time: 2min 2s


In [13]:
%%time
adata = adata[:,adata.var.highly_variable].copy()

CPU times: user 1min 57s, sys: 16.4 s, total: 2min 14s
Wall time: 1min 11s


## **Rechunking the Expression Matrix for Multi-GPU Execution**  

To optimize performance across **8 GPUs**, we rechunk the expression matrix (`adata.X`)  
so that each GPU processes an equal portion of the dataset.  

## **Scaling Gene Expression (Requires Synchronization)**  

To standardize gene expression values, we apply **feature scaling**,  
We also `persist` the results to ensure fast accessibility

In [14]:
n_rows = adata.shape[0]
n_cols = adata.shape[1]
rows_per_worker = (n_rows+7-1)//7
adata.X = adata.X.rechunk((rows_per_worker, n_cols)).persist()

In [15]:
%%time
rsc.pp.scale(adata, zero_center= False)

CPU times: user 1min 47s, sys: 5.93 s, total: 1min 53s
Wall time: 1min 52s


In [16]:
adata.X = adata.X.persist()

/home/scratch.sdicks_gpu/micromamba/envs/rapids-25.10/lib/python3.13/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 383.97 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


## **Principal Component Analysis (PCA) on GPU**  

To reduce dimensionality while preserving meaningful variation,  
we perform **Principal Component Analysis (PCA)** using **GPU acceleration**.

Finalizing the Transformation with `.compute()`
    * After computing the principal components, the data remains lazy (Dask CuPy array).
    * Calling `.compute()` on `adata.obsm["X_pca"]` performs the final transformation,
      projecting the data onto the computed PCs and materializing the result as a fully computed CuPy array.

In [17]:
%%time
rsc.pp.pca(adata, n_comps = 100,mask_var=None)

CPU times: user 4.59 s, sys: 660 ms, total: 5.25 s
Wall time: 4.75 s


In [18]:
%%time
adata.obsm["X_pca"] = rsc.get.X_to_CPU(adata.obsm["X_pca"])
adata.obsm["X_pca"] = adata.obsm["X_pca"].compute()

CPU times: user 1min 44s, sys: 22.7 s, total: 2min 7s
Wall time: 2min 8s


In [19]:
%%time
rsc.pp.neighbors(adata, n_pcs = 100,n_neighbors=15, algorithm="all_neighbors", algorithm_kwds={"algo":"nn_descent"})

CPU times: user 1h 59min 12s, sys: 8min 39s, total: 2h 7min 52s
Wall time: 3min 34s


In [20]:
%%time
rsc.tl.umap(adata)

CPU times: user 2min 51s, sys: 1min 3s, total: 3min 55s
Wall time: 2min 58s


In [21]:
%%time
rsc.tl.leiden(adata, resolution=1.0, use_dask=True)

/home/scratch.sdicks_gpu/micromamba/envs/rapids-25.10/lib/python3.13/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 29.29 GiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


[1763573682.208350] [umb-b300-dp-217:609610:0]            sock.c:495  UCX  ERROR bind(fd=210 addr=0.0.0.0:51282) failed: Address already in use


/home/scratch.sdicks_gpu/micromamba/envs/rapids-25.10/lib/python3.13/site-packages/cudf/core/reshape.py:397: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  warnings.warn(


CPU times: user 4min 47s, sys: 51.4 s, total: 5min 39s
Wall time: 4min 49s


In [22]:
print((time.time()-t1)/60)

18.71386560599009
